In [1]:
# NLP + K-Means Clustering

import pandas as pd
import re
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. Load dataset
df = pd.read_csv("ConCo-ref-SupAdmCo-unlinked.csv")

# 2. Select text column automatically
text_col = df.select_dtypes(include="object").columns[0]
df[text_col] = df[text_col].fillna("")

# 3. Clean text
def clean(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    return " ".join(text.split())

df["clean_text"] = df[text_col].apply(clean)

# 4. NLP: TF-IDF
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1,2)
)
X = tfidf.fit_transform(df["clean_text"])

# 5. Find best K using silhouette score
scores = {}

for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    scores[k] = silhouette_score(X, labels)

best_k = max(scores, key=scores.get)
print("Best K:", best_k)
print("Silhouette Score:", scores[best_k])

# 6. K-Means clustering
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X)

# 7. Cluster results
print("\nCluster sizes:")
print(df["Cluster"].value_counts().sort_index())

# 8. Top words in each cluster
words = tfidf.get_feature_names_out()

for i in range(best_k):
    top = kmeans.cluster_centers_[i].argsort()[-10:][::-1]
    print(f"\nCluster {i}:")
    print(", ".join(words[j] for j in top))

# 9. Save results
df.to_csv("ConCo-ref-SupAdmCo-unlinked_clustered.csv", index=False)

print("\nSaved successfully!")

/var/folders/88/w4w1n8l12kd_z42_6mrnndmw0000gn/T/ipykernel_5789/3881371352.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_col = df.select_dtypes(include="object").columns[0]
/Users/adnanaltimeemy/miniconda3/envs/coding/lib/python3.12/site-packages/sklearn/base.py:1403: ConvergenceWarning: Number of distinct clusters (3) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/Users/adnanaltimeemy/miniconda3/envs/coding/lib/python3.12/site-packages/sklearn/base.py:1403: ConvergenceWarning: Number o

Best K: 3
Silhouette Score: 0.998589562764457

Cluster sizes:
Cluster
0    676
1     32
2      1
Name: count, dtype: int64

Cluster 0:
txt, st txt, st, pl txt, pl

Cluster 1:
pl txt, pl, txt, st txt, st

Cluster 2:
st txt, st, txt, pl txt, pl

Saved successfully!


/Users/adnanaltimeemy/miniconda3/envs/coding/lib/python3.12/site-packages/sklearn/base.py:1403: ConvergenceWarning: Number of distinct clusters (3) found smaller than n_clusters (9). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/Users/adnanaltimeemy/miniconda3/envs/coding/lib/python3.12/site-packages/sklearn/base.py:1403: ConvergenceWarning: Number of distinct clusters (3) found smaller than n_clusters (10). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
